In [ ]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Gualberto"
STUDENT_ID = "TUPM-26-1130"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

In [ ]:
# [CELL 2] LOAD SECOM AND IDENTIFY USABLE FEATURES
WORK_DIR = Path.cwd()
DATA_DIR = next(
    (p for p in [WORK_DIR / "data", WORK_DIR.parent / "data"] if p.is_dir()),
    None
)
assert DATA_DIR is not None, "data/ folder not found (checked here and one level up)."
OUTPUT_DIR = WORK_DIR

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
outcomes = np.where(labels == -1, "Pass", "Fail")

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

print("Data Folder:", DATA_DIR.resolve())
print("Measurement Matrix:", raw_measurements.shape)
print("Pass Records:", (labels == -1).sum(), "| Fail Records:", (labels == 1).sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

In [ ]:
# [CELL 3] LABELED DATAFRAME, BALANCED SAMPLE AND IMPUTATION
df = pd.DataFrame(raw_measurements, columns=feature_names)
df.index.name = "Example_ID"
df["Result"] = labels
df["Outcome"] = outcomes

pass_total = int((labels == -1).sum())
fail_total = int((labels == 1).sum())
per_group = min(20 + ID_SUM, pass_total, fail_total)

rng2 = np.random.default_rng(RANDOM_SEED)

ex2_positions = rng2.choice(len(usable_indices), size=5, replace=False)
ex2_features = feature_names[usable_indices[ex2_positions]].tolist()

pass_ids = np.flatnonzero(labels == -1)
fail_ids = np.flatnonzero(labels == 1)
chosen_ids = np.concatenate([
    rng2.choice(pass_ids, size=per_group, replace=False),
    rng2.choice(fail_ids, size=per_group, replace=False),
])
chosen_ids.sort()

full_medians = df[ex2_features].median()

balanced = df.loc[chosen_ids, ex2_features + ["Result", "Outcome"]].copy()
missing_before = int(balanced[ex2_features].isna().sum().sum())
balanced[ex2_features] = balanced[ex2_features].fillna(full_medians)
balanced = balanced.reset_index()

balanced_path = OUTPUT_DIR / "balanced_sample.csv"
balanced.to_csv(balanced_path, index=False)

print("Pass / Fail Totals:", pass_total, "/", fail_total)
print("Records Per Outcome:", per_group)
print("Balanced Sample Size:", len(balanced))
print("Selected Features:", ex2_features)
print("Missing Before Imputation:", missing_before)
print("Missing After Imputation:", int(balanced[ex2_features].isna().sum().sum()))
print(balanced["Outcome"].value_counts())
print(balanced.head())

In [ ]:
# [CELL 4] GROUPED STATISTICS AND STANDARDIZED MEAN DIFFERENCE
grouped = balanced.groupby("Outcome")[ex2_features].agg(["count", "mean", "median", "std"])
print("Grouped Statistics:\n", grouped.round(4))

rows = []
for f in ex2_features:
    n_p = grouped.loc["Pass", (f, "count")]
    n_f = grouped.loc["Fail", (f, "count")]
    mean_p, mean_f = grouped.loc["Pass", (f, "mean")], grouped.loc["Fail", (f, "mean")]
    med_p, med_f = grouped.loc["Pass", (f, "median")], grouped.loc["Fail", (f, "median")]
    sd_p, sd_f = grouped.loc["Pass", (f, "std")], grouped.loc["Fail", (f, "std")]

    pooled_sd = np.sqrt(((n_p - 1) * sd_p**2 + (n_f - 1) * sd_f**2) / (n_p + n_f - 2))
    abs_diff = abs(mean_f - mean_p)
    abs_smd = abs_diff / pooled_sd if pooled_sd > 0 else 0.0

    rows.append({
        "Feature": f, "Pass_Count": int(n_p), "Fail_Count": int(n_f),
        "Pass_Mean": mean_p, "Fail_Mean": mean_f,
        "Pass_Median": med_p, "Fail_Median": med_f,
        "Pass_SD": sd_p, "Fail_SD": sd_f,
        "Abs_Mean_Diff": abs_diff, "Pooled_SD": pooled_sd,
        "Abs_Std_Mean_Diff": abs_smd,
    })

feature_comparison = (
    pd.DataFrame(rows)
    .sort_values("Abs_Std_Mean_Diff", ascending=False)
    .reset_index(drop=True)
)
feature_comparison.insert(0, "Rank", np.arange(1, len(feature_comparison) + 1))

comparison_path = OUTPUT_DIR / "feature_comparison.csv"
feature_comparison.to_csv(comparison_path, index=False)

print(feature_comparison.round(4))

In [ ]:
# [CELL 5] NUMPY-PANDAS VERIFICATION AND ASSESSMENT
lookup = feature_comparison.set_index("Feature")
np_pd_match = True
for f in ex2_features:
    p = balanced.loc[balanced["Outcome"] == "Pass", f].to_numpy()
    q = balanced.loc[balanced["Outcome"] == "Fail", f].to_numpy()

    np_values = [p.mean(), q.mean(), np.median(p), np.median(q),
                 p.std(ddof=1), q.std(ddof=1), abs(q.mean() - p.mean())]
    pd_values = [lookup.loc[f, c] for c in
                 ["Pass_Mean", "Fail_Mean", "Pass_Median", "Fail_Median",
                  "Pass_SD", "Fail_SD", "Abs_Mean_Diff"]]
    np_pd_match &= bool(np.allclose(np_values, pd_values))

    pooled = np.sqrt(((len(p) - 1) * p.var(ddof=1) + (len(q) - 1) * q.var(ddof=1)) / (len(p) + len(q) - 2))
    smd = abs(q.mean() - p.mean()) / pooled if pooled > 0 else 0.0
    np_pd_match &= bool(np.isclose(smd, lookup.loc[f, "Abs_Std_Mean_Diff"]))

best = feature_comparison.iloc[0]

print("--- Assessment Data: Exercise 2 ---")
print("Balanced Sample Size:", len(balanced))
print("Selected Features:", ex2_features)
print("Highest-Ranked Feature:", best["Feature"])
print("Absolute Standardized Mean Difference:", f"{best['Abs_Std_Mean_Diff']:.4f}")
print("NumPy-Pandas Statistics Match:", np_pd_match)
print(balanced_path.name, "Created:", balanced_path.is_file())
print(comparison_path.name, "Created:", comparison_path.is_file())